# Determine flagged company impact

### Inputs: 
- `"../derived/linkedin_company_urls_ai_validated.csv"`
- `"../data/derived/admissions.csv"`
- `"../data/derived/interviews.csv"`
- `"../data/derived/outcomes.csv"`

### Outputs:
- `"../derived/flagged_companies_positions.xlsx"`
- `"../derived/flagged_linked_company_urls.xlsx"`
  

### Purpose:

1. Clean and merge the original datasets.
2. Get list of all company names mentioned in the records.


After validation, a large number of company's LinkedIn URLs require manual correction. To make the process as effective as possible, we want to be able to sort the flagged company's by their impact, i.e., how many records are connected to that company.

In [1]:
import pandas as pd

In [3]:
companies = pd.read_csv("../data/derived/linkedin_company_urls_ai_validated.csv")
companies = companies[companies["Is Valid"] == False]
companies.sample(5)

,Company Name,LinkedIn URL,Reasoning,Is Valid
552,Vivaldi Partners,https://www.linkedin.com/company/vivaldi-group,**Reasoning:**\n\nThe LinkedIn URL provided is...,False
1067,State Farm Canada - Desjardins Insurance,https://ca.linkedin.com/company/desjardinsinsu...,**Reasoning:**\n\nThe LinkedIn URL provided (h...,False
946,MasterCard Advisors,https://www.linkedin.com/company/mastercard-da...,**Reasoning:**\n\nThe LinkedIn URL provided is...,False
472,GrabTaxi,https://www.linkedin.com/company/grabapp,**Final Assessment:**\n\n**Reasoning:** The co...,False
1422,"Fourth Battlefield Coordination Detachment, Un...",https://www.linkedin.com/company/us-army-force...,"Based on the information provided, the assessm...",False


In [4]:
admissions_data = pd.read_csv("../data/derived/admissions.csv")
interviews = pd.read_csv("../data/derived/interviews.csv")
outcomes = pd.read_csv("../data/derived/outcomes.csv")

records = pd.Series(
    admissions_data["Job 1 Organization"].tolist()
    + interviews["Employer"].tolist()
    + outcomes["Employer"].tolist(),
    name="Company",
)

record_count = records.value_counts()

In [5]:
def get_record_count(company_name: str):
    return record_count.loc[company_name]


companies["Affected Records"] = companies["Company Name"].map(get_record_count)

In [6]:
companies = companies.sort_values(by="Affected Records", ascending=False)

In [ ]:
companies["Affected Records"].value_counts()

Affected Records
1      227
2       84
3       14
4        7
5        5
6        5
8        3
186      1
7        1
11       1
14       1
239      1
17       1
18       1
22       1
24       1
27       1
58       1
89       1
91       1
178      1
16       1
Name: count, dtype: int64

In [ ]:
positions = []
for idx, row in companies[companies["Affected Records"] < 5].iterrows():
    pos = admissions_data.loc[
        admissions_data["Job 1 Organization"] == row["Company Name"], "Job 1 Title"
    ]

    if not pos.empty:
        pos = pos.to_frame()
        pos = pos.rename(columns={"Job 1 Title": "Job Title"})
        pos["Company"] = row["Company Name"]
        pos["Origin"] = "admissions"
        positions.append(pos)

    pos = interviews.loc[interviews["Employer"] == row["Company Name"], "Job Title"]

    if not pos.empty:
        pos = pos.to_frame()
        pos["Company"] = row["Company Name"]
        pos["Origin"] = "interviews"
        positions.append(pos)

    pos = outcomes.loc[outcomes["Employer"] == row["Company Name"], "Job Title"]

    if not pos.empty:
        pos = pos.to_frame()
        pos["Company"] = row["Company Name"]
        pos["Origin"] = "outcomes"
        positions.append(pos)

pd.concat(positions).to_excel("../derived/flagged_companies_positions.xlsx", index=None)

In [ ]:
companies.to_excel("../derived/flagged_linked_company_urls.xlsx", index=None)